In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym
from gymnasium import spaces

import collections

from dm_control import mujoco, viewer, suite
from dm_control.rl import control
from dm_control.suite import base, common
from dm_control.suite.utils import randomizers
from dm_control.utils import rewards
from dm_control.utils import io as resources
from dm_env import specs
import numpy as np
import os

from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
device

'cuda'

In [2]:

class MultiTaskPointMassMaze(base.Task):
    """A point_mass `Task` to reach target with smooth reward."""
    def __init__(self, random=None):
        super().__init__(random=random)

    def initialize_episode(self, physics):
        # randomizers.randomize_limited_and_rotational_joints(physics, self.random)
        physics.data.qpos[0] = 1
        physics.data.qpos[1] = 0

        super().initialize_episode(physics)

    def get_observation(self, physics):
        """Returns an observation of the state."""
        obs = collections.OrderedDict()
        obs['position'] = physics.position()
        obs['velocity'] = physics.velocity()
        return obs
    
    def get_reward_spec(self):
        return specs.Array(shape=(1,), dtype=np.float32, name='reward')

    def get_reward(self, physics):
        return 0



class Physics(mujoco.Physics):
    """physics for the point_mass domain."""

    def mass_to_target_dist(self, target):
        """Returns the distance from mass to the target."""
        d = target - self.named.data.geom_xpos['pointmass']
        return np.linalg.norm(d)

In [9]:
xml = resources.GetResource(f'mazes/antmaze_hardest.xml')
physics = Physics.from_xml_string(xml, common.ASSETS)
task = MultiTaskPointMassMaze()

dm_env = control.Environment(
    physics,
    task,
    time_limit=20,  
)



def random_policy(time_step):
    action_spec = dm_env.action_spec()
    return np.random.uniform(
        low=action_spec.minimum * 10,
        high=action_spec.maximum * 10,
        size=action_spec.shape
    )

# Launch viewer with the random policy
viewer.launch(dm_env, policy=random_policy)

---

In [4]:
import gym

/home/nazim/.local/lib/python3.10/site-packages/gym/wrappers/monitoring/video_recorder.py:9: DeprecationWarning: The distutils package is deprecated and slated for removal in Python 3.12. Use setuptools or check PEP 632 for potential alternatives
  import distutils.spawn


In [1]:
# import gym
# import d4rl 

# env = gym.make('antmaze-large-diverse-v2')

running build_ext
building 'mujoco_py.cymj' extension
x86_64-linux-gnu-gcc -Wno-unused-result -Wsign-compare -DNDEBUG -g -fwrapv -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -g -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2 -fPIC -I/home/nazim/.local/lib/python3.10/site-packages/mujoco_py -I/home/nazim/.mujoco/mujoco210/include -I/home/nazim/.local/lib/python3.10/site-packages/numpy/core/include -I/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/vendor/egl -I/usr/include/python3.10 -c /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/cymj.c -o /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/generated/_pyxbld_2.1.2.14_310_linuxgpuextensionbuilder/temp.linux-x86_64-3.10/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/cymj.o -fopenmp -w


/home/nazim/.local/lib/python3.10/site-packages/gym/wrappers/monitoring/video_recorder.py:9: DeprecationWarning: The distutils package is deprecated and slated for removal in Python 3.12. Use setuptools or check PEP 632 for potential alternatives
  import distutils.spawn
/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/builder.py:9: DeprecationWarning: The distutils.sysconfig module is deprecated, use sysconfig instead
  from distutils.sysconfig import customize_compiler


x86_64-linux-gnu-gcc -Wno-unused-result -Wsign-compare -DNDEBUG -g -fwrapv -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -g -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2 -fPIC -I/home/nazim/.local/lib/python3.10/site-packages/mujoco_py -I/home/nazim/.mujoco/mujoco210/include -I/home/nazim/.local/lib/python3.10/site-packages/numpy/core/include -I/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/vendor/egl -I/usr/include/python3.10 -c /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/gl/eglshim.c -o /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/generated/_pyxbld_2.1.2.14_310_linuxgpuextensionbuilder/temp.linux-x86_64-3.10/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/gl/eglshim.o -fopenmp -w


In file included from /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/gl/egl.h:39,
                 from /home/nazim/.local/lib/python3.10/site-packages/mujoco_py/gl/eglshim.c:2:
/home/nazim/.local/lib/python3.10/site-packages/mujoco_py/gl/eglplatform.h:99:10: fatal error: X11/Xlib.h: No such file or directory
   99 | #include <X11/Xlib.h>
      |          ^~~~~~~~~~~~
compilation terminated.


CompileError: command '/usr/bin/x86_64-linux-gnu-gcc' failed with exit code 1

In [31]:
gym.envs.registry

├──CartPole: [ v0, v1 ]
├──MountainCar: [ v0 ]
├──MountainCarContinuous: [ v0 ]
├──Pendulum: [ v1 ]
├──Acrobot: [ v1 ]
├──LunarLander: [ v2 ]
├──LunarLanderContinuous: [ v2 ]
├──BipedalWalker: [ v3 ]
├──BipedalWalkerHardcore: [ v3 ]
├──CarRacing: [ v1 ]
├──Blackjack: [ v1 ]
├──FrozenLake: [ v1 ]
├──FrozenLake8x8: [ v1 ]
├──CliffWalking: [ v0 ]
├──Taxi: [ v3 ]
├──Reacher: [ v2 ]
├──Pusher: [ v2 ]
├──InvertedPendulum: [ v2 ]
├──InvertedDoublePendulum: [ v2 ]
├──HalfCheetah: [ v2, v3 ]
├──Hopper: [ v2, v3 ]
├──Swimmer: [ v2, v3 ]
├──Walker2d: [ v2, v3 ]
├──Ant: [ v2, v3 ]
├──Humanoid: [ v2, v3 ]
└──HumanoidStandup: [ v2 ]

---

In [10]:
from gymnasium_robotics.envs.maze.maze_v4 import Maze

In [12]:
LARGE_MAZE = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1],
    [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]

Maze.make_maze(
    'mazes/ant.xml', 
    LARGE_MAZE, 
    maze_size_scaling=4, 
    maze_height=0.5
)

(<gymnasium_robotics.envs.maze.maze_v4.Maze at 0x788e8ae975e0>,
 '/tmp/ant_maze1747918868.7368279.xml')